# **TTS And STT with Traffic Tool Integration**



This system is a voice-based smart traffic assistant that combines Speech-to-Text (STT), Text-to-Speech (TTS), and a traffic route tool into one integrated application.

The system allows users to speak their route queries, such as distance or travel time between two locations. The microphone records the user’s voice, and the STT module (Whisper model) converts the speech into text. The converted text is then processed by the AI assistant. If the query is related to route, distance, or travel time, the assistant automatically calls the traffic tool, which uses the OpenRouteService API to fetch real-time distance and estimated travel time.

After receiving the route data, the assistant generates a proper response in text format. Finally, the TTS module converts the text response into natural-sounding speech and plays it back to the user.


## **System Flow**

    1.User speaks a route query

    2.STT converts speech → text

    3.AI detects route intent

    4.Traffic tool fetches distance and travel time

    5.Assistant generates response

    6.TTS converts response → voice output

This integration creates a complete end-to-end voice navigation assistant that supports hands-free interaction and provides accurate route information with spoken responses.

## **Installation**
Install the required libraries for API communication and audio processing.

In [ ]:
!pip install openai requests -q

## **Configuration and Imports**
Import necessary libraries and initialize the OpenAI client.

In [ ]:
from openai import OpenAI
import requests
import json
from google.colab import userdata
from IPython.display import Audio, display


client = OpenAI(
    api_key=userdata.get("api_key"),
    base_url=userdata.get("base_url")
)

ORS_API_KEY = userdata.get("ors_api_key")



## **Conversation Memory Setup**
Initialize the system prompt to define the assistant's behavior.

In [ ]:
memory = [{
    "role": "system",
    "content": """You are a smart voice assistant.

If the user asks about:
- route
- road
- direction
- distance
- travel time
- how to go
- going to somewhere

ALWAYS call the get_traffic tool.

If only one place is mentioned, politely ask for the starting location.

Never answer route questions without using the tool."""
}]

## **Traffic Tool Implementation**
Defines the `get_traffic` function using OpenRouteService.

In [ ]:
# -----------------------------
# TRAFFIC TOOL (FREE API)
# -----------------------------

def get_traffic(source, destination):

    geocode_url = "https://api.openrouteservice.org/geocode/search"

    source_geo = requests.get(
        geocode_url,
        params={"api_key": ORS_API_KEY, "text": source}
    ).json()

    dest_geo = requests.get(
        geocode_url,
        params={"api_key": ORS_API_KEY, "text": destination}
    ).json()

    if not source_geo.get("features") or not dest_geo.get("features"):
        return "Could not find one of the locations."

    source_coords = source_geo["features"][0]["geometry"]["coordinates"]
    dest_coords = dest_geo["features"][0]["geometry"]["coordinates"]

    directions_url = "https://api.openrouteservice.org/v2/directions/driving-car"

    headers = {
        "Authorization": ORS_API_KEY,
        "Content-Type": "application/json"
    }

    body = {
        "coordinates": [source_coords, dest_coords]
    }

    response = requests.post(directions_url, headers=headers, json=body)
    data = response.json()

    if "routes" not in data:
        return "Unable to fetch route information."

    summary = data["routes"][0]["summary"]

    distance_km = round(summary["distance"] / 1000, 2)
    duration_min = round(summary["duration"] / 60, 2)

    return (
        f"Distance from {source} to {destination} is {distance_km} kilometers. "
        f"Estimated travel time is {duration_min} minutes."
    )


In [ ]:

# -----------------------------
# TOOL MAPPING
# -----------------------------

tool_functions = {
    "get_traffic": get_traffic
}



## **Tool Definition for OpenAI**
Map the function to a tool definition for the model.

In [ ]:
# -----------------------------
# TOOL DEFINITION
# -----------------------------

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_traffic",
            "description": "Get distance and travel time between two locations",
            "parameters": {
                "type": "object",
                "properties": {
                    "source": {"type": "string"},
                    "destination": {"type": "string"}
                },
                "required": ["source", "destination"]
            }
        }
    }
]



## **Text-to-Speech (TTS) Integration**
Convert assistant responses into audio.

In [ ]:
# -----------------------------
# TEXT TO SPEECH FUNCTION
# -----------------------------

def text_to_speech(text, filename="assistant_response.wav"):

    response = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )

    audio_bytes = response.read()

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    display(Audio(filename, autoplay=True))



##  **Agent Logic**
Core loop handling chat completions and tool calls.

In [ ]:
# -----------------------------
# AGENT FUNCTION
# -----------------------------

def run_agent(user_text):

    memory.append({"role": "user", "content": user_text})

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=memory,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls:

        tool_call = message.tool_calls[0]
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        memory.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_name,
                        "arguments": tool_call.function.arguments
                    }
                }
            ]
        })

        result = tool_functions[tool_name](**arguments)

        memory.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

        second_response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=memory
        )

        final_text = second_response.choices[0].message.content
        memory.append({"role": "assistant", "content": final_text})

    else:
        final_text = message.content
        memory.append({"role": "assistant", "content": final_text})

    return final_text

## **Main Interactive Loop**
Start chatting with the voice assistant. Type 'quit' to exit.

In [ ]:
# Main loop

while True:

    user_input = input("\nEnter your question (or quit): ")

    if user_input.lower() == "quit":
        break

    # Play user input as audio
    print("User:", user_input)
    text_to_speech(user_input, "user_audio.wav")

    # Run agent
    response = run_agent(user_input)

    print("Assistant:", response)

    # Play assistant response as audio
    text_to_speech(response, "assistant_audio.wav")


Enter your question (or quit): coimbatore to erode
User: coimbatore to erode


Assistant: The distance from Coimbatore to Erode is approximately 97 kilometers, and it usually takes about 78 minutes to drive there.



Enter your question (or quit): quit


# **Speech-to-Text (STT) Integration**

In [ ]:
import os
import base64
from google.colab import output
from IPython.display import Javascript

## **Speech-To-Text Function**

This function converts a recorded audio file into text using the OpenAI Whisper model.

In [ ]:
def speech_to_text(audio_file_path):

    try:
        with open(audio_file_path, "rb") as audio_file:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                language="en"
            )

        text = transcript.text.strip()

        if not text or len(text) < 2:
            return None

        return text

    except Exception as e:
        print("STT Error:", e)
        return None

## **Audio Recording Function**

This function records audio from the browser microphone inside Google Colab and saves it as a .wav file.

In [ ]:
def record_audio(filename="input.wav", seconds=5):

    if os.path.exists(filename):
        os.remove(filename)

    print(f"🎤 Recording for {seconds} seconds... Allow microphone access.")

    audio_data = output.eval_js(f"""
        new Promise(async (resolve) => {{
            const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
            const recorder = new MediaRecorder(stream);
            const chunks = [];

            recorder.ondataavailable = e => chunks.push(e.data);

            recorder.start();

            setTimeout(() => {{
                recorder.stop();
            }}, {seconds} * 1000);

            recorder.onstop = async () => {{
                const blob = new Blob(chunks);
                const arrayBuffer = await blob.arrayBuffer();
                const base64String = btoa(
                    String.fromCharCode(...new Uint8Array(arrayBuffer))
                );
                stream.getTracks().forEach(track => track.stop());
                resolve(base64String);
            }};
        }})
    """)

    audio_bytes = base64.b64decode(audio_data)

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    print("✅ Recording finished.\n")

## **Main Interactive Loop**
Start chatting with the voice assistant. Type 'quit' to exit.

In [ ]:
while True:

    command = input("\nClick 's' to record voice or type 'quit': ").lower()

    if command == "quit":
        break

    if command == "s":

        print("🎤 Recording... Speak now!")
        record_audio("input.wav", seconds=5)

        user_text = speech_to_text("input.wav")

        if user_text is None:
            print("⚠ No speech detected. Try again.")
            continue

        print("🗣 User:", user_text)

        response = run_agent(user_text)

        print("🤖 Assistant:", response)

        text_to_speech(response, "assistant_audio.wav")

    else:
        print("Invalid input. Please press 's' or type 'quit'.")


Click 's' to record voice or type 'quit': s
🎤 Recording... Speak now!
🎤 Recording for 5 seconds... Allow microphone access.
✅ Recording finished.

🗣 User: Can you tell me the route for Chennai to Coimbatore?
🤖 Assistant: The distance from Chennai to Coimbatore is approximately 500 kilometers, and the estimated travel time is around 377 minutes. Would you like directions or information on the best route?



Click 's' to record voice or type 'quit': s
🎤 Recording... Speak now!
🎤 Recording for 5 seconds... Allow microphone access.
✅ Recording finished.

🗣 User: Right
🤖 Assistant: Great! If you need directions or any other assistance, just let me know. Safe travels!



Click 's' to record voice or type 'quit': s
🎤 Recording... Speak now!
🎤 Recording for 5 seconds... Allow microphone access.
✅ Recording finished.

🗣 User: What is the distance between Bangalore to Chennai?
🤖 Assistant: The distance from Bangalore to Chennai is approximately 321 kilometers, and the estimated travel time is about 245 minutes. Would you like directions or travel tips?



Click 's' to record voice or type 'quit': quit


## **Observations**


### **Text-to-Speech (TTS)**

* The TTS system provides clear and proper voice output for the assistant’s response.

* The pronunciation of city names like Coimbatore and Erode is correct.

* Distance and time values are spoken clearly and understandably.

* The audio playback works without delay or errors.

* The generated voice sounds natural and easy to listen to.

* Overall, the TTS system produces accurate and clear spoken output for route queries.

### **Speech-to-Text (STT)**

* The accuracy is sometimes low because the microphone does not always record the user input clearly.
* Occasionally, the system recognizes the wrong word instead of what the user actually said.
* If the user speaks slowly or starts late, some words may get cut due to the 5-second limit.
* Background noise can reduce transcription accuracy.
* Short or unclear speech may not be detected properly.
* Overall, the system works well for clear speech but may give small errors in real-time voice input.